In [1]:
from pathlib import Path

from noid_rebuild import (
    NoidRebuilder,
    build_manifest,
    load_csv_entries,
    verify_database,
)

CSV_PATH = Path("../csv2JSON/aardvark_export_2026-09-14.csv").resolve()

entries, row_errors = load_csv_entries(CSV_PATH)

assert not row_errors, row_errors[:10]

print(f"Validated CSV entries: {len(entries)}")

Validated CSV entries: 930


In [2]:
LIVE_ONLY_NOIDS = {
    "77981/gmgs15dv46t",
    "77981/gmgs2rbp036",
    "77981/gmgs3xsj48c",
    "77981/gmgs4xgxdk4",
    "77981/gmgs7pvmdn2",
    "77981/gmgscc2fs30",
    "77981/gmgsh44j2g5",
    "77981/gmgsmw6mbvg",
    "77981/gmgsrn8pn7p",
}

manifest = build_manifest(
    entries,
    hold_only_noids=LIVE_ONLY_NOIDS,
)

print(f"Bound records: {len(manifest.entries)}")
print(f"Total holds: {len(manifest.hold_noids)}")
print(f"Bindings: {len(manifest.bindings)}")
print(f"Hold-only records: {len(LIVE_ONLY_NOIDS)}")

Bound records: 930
Total holds: 939
Bindings: 5561
Hold-only records: 9


In [3]:
NOID_BINARY = Path("perl5/bin/noid").resolve()
PERL5LIB = Path("perl5/lib/perl5").resolve()
DATABASE_DIR = Path("scratch/rebuild-clean").resolve()

assert NOID_BINARY.is_file()
assert (DATABASE_DIR / "NOID" / "noid.bdb").is_file()

rebuilder = NoidRebuilder(
    noid_binary=NOID_BINARY,
    database_dir=DATABASE_DIR,
    perl5lib=PERL5LIB,
)

database_dump = rebuilder.dump()
verify_database(manifest, database_dump)

print("Scratch database exactly matches the reconstruction manifest.")

Scratch database exactly matches the reconstruction manifest.


In [4]:
import importlib
import noid_rebuild

importlib.reload(noid_rebuild)

from noid_rebuild import serialize_manifest

In [8]:
from noid_rebuild import serialize_manifest

manifest_text, manifest_sha256 = serialize_manifest(manifest)

ARTIFACT_DIR = Path("scratch/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = ARTIFACT_DIR / "agsl-noid-rebuild-manifest-v1.json"
checksum_path = ARTIFACT_DIR / f"{manifest_path.name}.sha256"

manifest_path.write_text(manifest_text, encoding="utf-8")
checksum_path.write_text(
    f"{manifest_sha256}  {manifest_path.name}\n",
    encoding="utf-8",
)

print(f"Manifest: {manifest_path}")
print(f"SHA-256: {manifest_sha256}")

Manifest: scratch/artifacts/agsl-noid-rebuild-manifest-v1.json
SHA-256: 5acd9066a7cc395dd047e45e8c4087cd0d6bed23951d397c2b14bedf94b29628


In [10]:
importlib.reload(noid_rebuild)

from noid_rebuild import load_manifest_file

portable_manifest = load_manifest_file(
    manifest_path,
    expected_sha256=manifest_sha256,
)

assert portable_manifest.entries == manifest.entries
assert portable_manifest.hold_noids == manifest.hold_noids
assert portable_manifest.bindings == manifest.bindings

print("Portable manifest reloaded and fully validated.")
print(f"Holds: {len(portable_manifest.hold_noids)}")
print(f"Bindings: {len(portable_manifest.bindings)}")

Portable manifest reloaded and fully validated.
Holds: 939
Bindings: 5561


In [11]:
importlib.reload(noid_rebuild)

from noid_rebuild import serialize_noid_commands

command_text, command_sha256 = serialize_noid_commands(portable_manifest)

command_path = ARTIFACT_DIR / "agsl-noid-rebuild-v1.noid"
command_checksum_path = ARTIFACT_DIR / f"{command_path.name}.sha256"

command_path.write_text(command_text, encoding="utf-8")
command_checksum_path.write_text(
    f"{command_sha256}  {command_path.name}\n",
    encoding="utf-8",
)

assert len(command_text.splitlines()) == 6500

print(f"Commands: {command_path}")
print("Operations: 6500")
print(f"SHA-256: {command_sha256}")

Commands: scratch/artifacts/agsl-noid-rebuild-v1.noid
Operations: 6500
SHA-256: 44be757bead8f6b65baff7700f47e0247ff2f22c046249bef8c522f405e0d5dc


In [14]:
import subprocess
import os

bulk_test_db = Path("scratch/native-command-test").resolve()
assert not (bulk_test_db / "NOID").exists()

bulk_test_db.mkdir(parents=True, exist_ok=True)

create_result = subprocess.run(
    [
        str(NOID_BINARY),
        "-f",
        str(bulk_test_db),
        "dbcreate",
        "gmgs.reeeeeek",
        "long",
        "77981",
        "University of Wisconsin-Milwaukee Libraries",
        "gmgs",
    ],
    env={**os.environ, "PERL5LIB": str(PERL5LIB)},
    capture_output=True,
    text=True,
    check=True,
)

print("Fresh native-command test database created.")

Fresh native-command test database created.


In [15]:
with command_path.open("r", encoding="utf-8") as command_stream:
    native_result = subprocess.run(
        [
            str(NOID_BINARY),
            "-f",
            str(bulk_test_db),
            "-",
        ],
        env={**os.environ, "PERL5LIB": str(PERL5LIB)},
        stdin=command_stream,
        capture_output=True,
        text=True,
        check=True,
    )

hold_successes = native_result.stdout.count("ok: 1 hold placed")
bind_successes = native_result.stdout.count("Status:  ok")

assert hold_successes == 939
assert bind_successes == 5561
assert not native_result.stderr.strip(), native_result.stderr

print(f"Native holds accepted: {hold_successes}")
print(f"Native bindings accepted: {bind_successes}")

Native holds accepted: 939
Native bindings accepted: 5561


In [16]:
native_test_rebuilder = NoidRebuilder(
    noid_binary=NOID_BINARY,
    database_dir=bulk_test_db,
    perl5lib=PERL5LIB,
)

native_test_dump = native_test_rebuilder.dump()
verify_database(portable_manifest, native_test_dump)

assert len(native_test_dump.holds) == 939
assert len(native_test_dump.bindings) == 5561

print("The native command file reproduced the manifest exactly.")

The native command file reproduced the manifest exactly.


In [17]:
from noid_rebuild import parse_database_dump

replacement_dump_path = ARTIFACT_DIR / "replacement.dump"
replacement_dump = parse_database_dump(
    replacement_dump_path.read_text(encoding="utf-8")
)

verify_database(portable_manifest, replacement_dump)

assert len(replacement_dump.holds) == 939
assert len(replacement_dump.bindings) == 5561

print("Production-built replacement exactly matches the manifest.")


Production-built replacement exactly matches the manifest.
